# *Feature_Selection*

In [1]:
from Constants import PATH
PATH=PATH
import os
os.chdir(PATH)
from utils_packeges import *

### *laod_cleaned_data*

In [2]:
from utils import laod_cleaned_data

df,numerical_cols,categorical_cols,boolean_cols,date_cols=laod_cleaned_data()

print(f"numerical_cols : \n{numerical_cols}")
print("--------------------------------------------------")
print(f"categorical_cols : \n{categorical_cols}")
print("--------------------------------------------------")
print(f"boolean_cols : \n{boolean_cols}")
print("--------------------------------------------------")
print(f"date_cols : \n{date_cols}")

numerical_cols : 
['vietnam_season', 'price', 'total_volume', 'brazil', 'india', 'vietnam', 'indonesia', 'china', 'jordan_max_price', 'jordan_min_price', 'demand', 'supply', 'month', 'year', 'DayOfMonth']
--------------------------------------------------
categorical_cols : 
['p_color']
--------------------------------------------------
boolean_cols : 
['brazil_season', 'indonesia_season', 'india_season', 'china_season']
--------------------------------------------------
date_cols : 
['week_start_dt', 'week_end_dt']


In [4]:
from sklearn.ensemble import RandomForestRegressor
from greedy_boruta import GreedyBorutaPy


X = df.drop(columns=["price", "week_start_dt", "week_end_dt"],axis=1)
y = df["price"]
X = pd.get_dummies(X, columns=categorical_cols, drop_first=True, dummy_na=True)

rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

selector = GreedyBorutaPy(
    rf,
    n_estimators='auto',   # or 'split'
    verbose=1,
    random_state=42
)

selector.fit(X.values, y.values)

# Results
confirmed_features = X.columns[selector.support_].tolist()
tentative_features = X.columns[selector.support_weak_].tolist() if hasattr(selector, 'support_weak_') else []

print("Confirmed important:", confirmed_features)
print("Tentative:", tentative_features)
print("Rejected:", X.columns[~selector.support_].tolist())

Iteration: 1 / 9
Iteration: 2 / 9
Iteration: 3 / 9
Iteration: 4 / 9
Iteration: 5 / 9
Iteration: 6 / 9
Iteration: 7 / 9
Iteration: 8 / 9


GreedyBorutaPy finished running.

Iteration: 	9 / 9
Confirmed: 	7
Tentative: 	0
Rejected: 	14
Confirmed important: ['total_volume', 'brazil', 'vietnam', 'indonesia', 'jordan_max_price', 'jordan_min_price', 'demand']
Tentative: []
Rejected: ['vietnam_season', 'india', 'china', 'brazil_season', 'indonesia_season', 'india_season', 'china_season', 'supply', 'month', 'year', 'DayOfMonth', 'p_color_red', 'p_color_yellow', 'p_color_nan']


In [5]:
selected_features_1=pd.concat([X[confirmed_features].astype("int"),y],axis=1)
selected_features_1

,total_volume,brazil,vietnam,indonesia,jordan_max_price,jordan_min_price,demand,price
0,1596040,10793,1519588,0,6,6,0,6.599075
1,1596040,10793,1519588,0,7,7,0,7.175335
2,1596040,10793,1519588,0,7,7,16,7.300575
3,2295578,5677,2274625,0,7,7,271,7.379675
4,2295578,5677,2274625,0,7,7,42,7.175335
...,...,...,...,...,...,...,...,...
1210,2761128,8695,2530027,180220,7,7,88,7.334644
1211,2761128,8695,2530027,180220,9,9,305,9.008137
1212,2665343,167,2521054,78334,7,7,97,7.259712
1213,2665343,167,2521054,78334,7,6,102,6.910027


In [6]:
selected_features_1.to_csv("Data_Sets/selected_features_boruta.csv",index=False)